In [1]:
faq_path = "databases/01_FAQ.csv"
stubs_path = "databases/02_Stubs.csv"

import pandas as pd

faq_csv = pd.read_csv(faq_path, delimiter="|")
faq_csv.head()

,category,question,answer
0,FAQ: Общие вопросы,Скрипт ответа на предложения пользователя/вопр...,Мы постоянно работаем над улучшением платформы...
1,FAQ: Общие вопросы,Скрипт ответа на нецензурную неконструктивную ...,Мы всегда рады конструктивной критике и очень ...
2,FAQ: Общие вопросы,Скрипт для вежливой неконструктивной негативно...,"Мы всегда рады конструктивной критике, поэтому..."
3,FAQ: Общие вопросы,Скрипт ответа на случай глобального сбоя на пл...,На данный момент на платформе наблюдаются техн...
4,FAQ: Общие вопросы,Скрипт ответа по вопросу насчет конкурса для с...,Мы рады всем авторам на нашей площадке и стара...


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_deepseek import ChatDeepSeek


llm = ChatDeepSeek(
    model="deepseek-reasoner",
    temperature=0.5,
    top_p=0.6,
    max_tokens=None,
    timeout=None,
    max_retries=10,
    # other params...
)

llm

ChatDeepSeek(client=<openai.resources.chat.completions.completions.Completions object at 0x73b02d1e9d00>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x73b02d1e8680>, root_client=<openai.OpenAI object at 0x73b02d570200>, root_async_client=<openai.AsyncOpenAI object at 0x73b02da8d040>, model_name='deepseek-reasoner', temperature=0.5, model_kwargs={}, max_retries=10, top_p=0.6, api_key=SecretStr('**********'), api_base='https://api.deepseek.com/v1')

In [4]:
template = '''
You are a regular user of the Rutube video platform, where you watch movies, series, cartoons, shows, and live broadcasts.
You’ve just received the following informational text in Russian (about Rutube's features, content, updates, or services).

Think like an actual user reading this message: what would you want to know or clarify after reading it?

Come up with 5 natural and realistic questions in the first person that could be answered directly by the content of the text.
Avoid overly generic or technical questions—focus on what a real user would likely ask in this situation.

Output ONLY the questions in valid JSON format, following this structure:
{
  "questions": [
    "Первый вопрос",
    "Второй вопрос",
    "Третий вопрос",
    "Четвертый вопрос",
    "Пятый вопрос"
  ]
}
'''

In [5]:
template_filter = lambda question, answer: f'''
You are a technical support specialist for the Rutube video platform.
The user asked the following question in Russian:
{question}
The chatbot responded with the following answer in Russian:
{answer}

Your task is to evaluate whether the chatbot's response is appropriate and helpful to the user's question.
Follow these rules:
    Answer only "Yes" or "No"

    If the user's question involves internal or confidential information, and the chatbot correctly avoids disclosing it, respond with "Yes" — this is the correct behavior.

    If the question does not involve internal or confidential information, but the chatbot refuses to answer by citing confidentiality, respond with "No" — the response is unnecessarily evasive.

    If the question does not involve internal information, and the chatbot responds accurately, clearly, and helpfully, respond with "Yes".

    Respond with "No" if the chatbot's answer is incorrect, off-topic, unclear, or unhelpful.
'''

In [6]:
import time
import json
import asyncio
augmented_questions = []
counter = 0


async def filter_question(question, answer):
    return await asyncio.to_thread(lambda: "yes" in llm.invoke(template_filter(question, answer)).text().lower())

for row in faq_csv.itertuples(index=True):
    begin = time.time()
    id = row.Index
    question, answer = row.question, row.answer
    prompt = template + "\n" + question + "." + answer
    # Filter questions concurrently
    questions = json.loads((await llm.ainvoke(prompt)).text())["questions"]
    filters = await asyncio.gather(*[filter_question(q, answer) for q in questions])
    filtered_questions = [q for q, keep in zip(questions, filters) if keep]
    augmented_questions.append({"id": id, "questions": filtered_questions})
    end = time.time()
    print(f"Finished iteration for id={id} in {end - begin} seconds")

['Как узнать, когда моё пожелание будет реализовано?']
Finished iteration for id=0 in 36.00231075286865 seconds


JSONDecodeError: Expecting value: line 1 column 1 (char 0)